# SAE reconstruction logit lens: did the sparse code change what the model sees?

For CLIP and SigLIP1 (which have a text space) we run the text-similarity logit
lens on the original patch activations and on the SAE-reconstructed activations,
side by side, per layer, and report per-patch label agreement. The supervised ViT
has no text encoder, so it uses an ImageNet class-logit readout (agreement number
only, no emoji grid).

Uses the topk k=128 SAEs trained in the comparison run (checkpoints in Drive).


## 1 — Install + download SAE checkpoints from Drive


In [ ]:
# Local run: your ViT-Prisma kernel already has the deps. Uncomment only if needed.
# !pip install -q transformers==4.44.2 einops timm datasets huggingface_hub tqdm gdown
# !pip install -q git+https://github.com/asharalam11/ViT-Prisma.git@add_siglip2

import os
# checkpoints live under ./content/saes/<model>/  (a 'content' dir in the current working dir)
CKPT_DIR = os.path.join(os.getcwd(), 'content', 'saes')
os.makedirs(CKPT_DIR, exist_ok=True)

# To fetch from Drive, paste each model's FOLDER ID (the last path segment of the
# folder URL: drive.google.com/drive/folders/<THIS_ID>). Leave blank to skip a model
# (e.g. if its .pt files are already in ./content/saes/<model>/).
FOLDER_IDS = {
    'clip':   '',   # id of the sae_ckpt_clip_topk_128 folder
    'siglip': '',   # id of the sae_siglip_ckpt_topk_128 folder
    'vit':    '',   # id of the sae_ckpt_vit_topk_128 folder
}
for name, fid in FOLDER_IDS.items():
    dest = os.path.join(CKPT_DIR, name)
    if fid:
        url = f'https://drive.google.com/drive/folders/{fid}'
        print('downloading', name, 'from', url)
        !gdown --folder "{url}" -O "{dest}" --quiet
    else:
        print(f'{name}: no folder id set, expecting .pt files already in {dest}')
    if os.path.isdir(dest):
        print('  ', name, 'contains:', os.listdir(dest)[:3], '...')


## 2 — Load models, tokenizers, text embeddings, and the logit-lens helpers


In [ ]:
import torch, glob, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw, ImageFont
from torchvision import transforms
from transformers import CLIPModel, CLIPTokenizer, SiglipModel, AutoTokenizer
from vit_prisma.models.model_loader import load_hooked_model
from vit_prisma.sae import VisionModelSAERunnerConfig
from vit_prisma.sae.sae import StandardSparseAutoencoder

device = 'cuda' if torch.cuda.is_available() else 'cpu'
LAYERS = [0, 4, 8, 11]

# where checkpoints live locally (defined here so it does not depend on the download cell)
CKPT_DIR = os.path.join(os.getcwd(), 'content', 'saes')

# --- label set with emojis (same as the raw logit-lens demo) ---
LABELS = [
    ('🐕', 'dog',        'a photo of a dog',                        '#8B4513'),
    ('🐈', 'cat',        'a photo of a cat',                        '#808080'),
    ('👁', 'eye',        'a close-up of an eye',                    '#4169E1'),
    ('👄', 'nose/mouth', 'a close-up of a nose or mouth',           '#DC143C'),
    ('🟧', 'fur',        'animal fur texture',                      '#D2691E'),
    ('🌿', 'bg',         'background, grass, sky, or plain surface','#A8C8A0'),
    ('🐾', 'paw',        'an animal paw or leg',                    '#556B2F'),
]
label_emojis = [l[0] for l in LABELS]
label_names  = [l[1] for l in LABELS]
label_texts  = [l[2] for l in LABELS]
label_colors = [l[3] for l in LABELS]

# Apple Color Emoji (macOS); falls back to None (solid colors only) elsewhere
_EMOJI_FONT = None
for _fp in ['/System/Library/Fonts/Apple Color Emoji.ttc']:
    try:
        _EMOJI_FONT = ImageFont.truetype(_fp, 32); break
    except OSError:
        pass

def norm_text(embs):
    return embs / embs.norm(dim=-1, keepdim=True)

def patch_label_indices(activations, text_embs, ln, projection=None):
    acts = ln(activations.float())
    if projection is not None:
        acts = acts @ projection.T
    acts = acts / acts.norm(dim=-1, keepdim=True)
    return (acts @ text_embs.T).argmax(dim=-1)


## 3 — Image + per-model preprocessing


In [ ]:
import requests
_candidates = ['../demos/cat_dog.png', 'demos/cat_dog.png', 'content/cat_dog.png', 'cat_dog.png']
img = None
for _p in _candidates:
    if os.path.exists(_p):
        img = Image.open(_p).convert('RGB'); print('image:', _p); break
if img is None:
    img = Image.open(requests.get(
        'https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg',
        stream=True).raw).convert('RGB')
    print('image: fetched from web')
img_224 = img.resize((224, 224))

def make_x(mean, std):
    pp = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(),
                             transforms.Normalize(mean=mean, std=std)])
    return pp(img).unsqueeze(0).to(device)

x_clip   = make_x([0.48145466,0.4578275,0.40821073], [0.26862954,0.26130258,0.27577711])
x_siglip = make_x([0.5,0.5,0.5], [0.5,0.5,0.5])

# ViT (timm) uses its own normalization
import timm
_pc = timm.get_pretrained_cfg('vit_base_patch16_224')
x_vit = make_x(list(_pc.mean), list(_pc.std))


## 4 — SAE loader (rebuild config, load state dict)


In [ ]:
def load_sae(model_key, layer, filename_tag):
    path = os.path.join(CKPT_DIR, model_key, f'sae_*{filename_tag}_layer{layer}.pt')
    hits = glob.glob(path)
    if not hits:
        raise FileNotFoundError(path)
    cfg = VisionModelSAERunnerConfig(
        model_name='x', model_class_name='HookedViT',
        hook_point_layer=layer, d_in=768, expansion_factor=4,
        context_size=196, image_size=224,
        activation_fn_str='topk', activation_fn_kwargs={'k': 128},
        normalize_activations='layer_norm', architecture='standard',
        log_to_wandb=False, _device=device,
    )
    sae = StandardSparseAutoencoder(cfg).to(device)
    sae.load_state_dict(torch.load(hits[0], map_location=device))
    sae.eval()
    return sae

@torch.no_grad()
def reconstruct(sae, acts):
    return sae(acts.to(device))[0]   # sae_out is the reconstruction


## 5 — CLIP and SigLIP: original vs SAE-reconstructed logit lens

Filename tags: CLIP checkpoints end `clip_topk_128_layer{N}.pt`,
SigLIP end `topk_128_layer{N}.pt`. Adjust `TAG` below if yours differ.


In [ ]:
# load full models + text embeddings
clip_vision = load_hooked_model('openai/clip-vit-base-patch16', device=device).to(device); clip_vision.eval()
clip_full   = CLIPModel.from_pretrained('openai/clip-vit-base-patch16').to(device)
clip_tok    = CLIPTokenizer.from_pretrained('openai/clip-vit-base-patch16')
clip_proj   = clip_full.visual_projection.weight.detach()
with torch.no_grad():
    clip_text = norm_text(clip_full.get_text_features(**clip_tok(label_texts, return_tensors='pt', padding=True).to(device)))

siglip_vision = load_hooked_model('google/siglip-base-patch16-224', device=device).to(device); siglip_vision.eval()
siglip_full   = SiglipModel.from_pretrained('google/siglip-base-patch16-224').to(device)
siglip_tok    = AutoTokenizer.from_pretrained('google/siglip-base-patch16-224', use_fast=False)
with torch.no_grad():
    siglip_text = norm_text(siglip_full.get_text_features(**siglip_tok(label_texts, return_tensors='pt', padding='max_length', max_length=64).to(device)))

with torch.no_grad():
    _, cache_clip = clip_vision.run_with_cache(x_clip)
    _, cache_sig  = siglip_vision.run_with_cache(x_siglip)

def agreement(a, b):
    return float((a == b).float().mean())


In [ ]:
def _hex(h): h=h.lstrip('#'); return tuple(int(h[i:i+2],16) for i in (0,2,4))

def draw_grid(ax, base_img, indices, colors, emojis, n=14, title='', scale=4):
    ps   = 224 // n           # 16
    sz   = 224 * scale        # 896
    ps_s = ps * scale         # 64
    canvas  = base_img.resize((sz, sz), Image.LANCZOS).convert('RGBA')
    overlay = Image.new('RGBA', (sz, sz), (0, 0, 0, 0))
    d = ImageDraw.Draw(overlay)
    font = ImageFont.truetype(_EMOJI_FONT.path, ps_s // 2) if _EMOJI_FONT else None
    idx = indices.reshape(n, n).cpu().numpy()
    for r in range(n):
        for c in range(n):
            k = int(idx[r, c])
            x0, y0 = c * ps_s, r * ps_s
            d.rectangle([x0, y0, x0 + ps_s, y0 + ps_s], fill=_hex(colors[k]) + (90,))
            if font is not None:
                d.text((x0 + ps_s // 4, y0 + ps_s // 5), emojis[k], font=font, embedded_color=True)
    ax.imshow(Image.alpha_composite(canvas, overlay))
    ax.set_title(title, fontsize=9); ax.axis('off')

for model_key, cache, vision, text, proj, tag, drop_cls in [
    ('clip',   cache_clip, clip_vision,   clip_text,   clip_proj, 'clip_topk_128', True),
    ('siglip', cache_sig,  siglip_vision, siglip_text, None,      'topk_128',      False),
]:
    fig, axes = plt.subplots(len(LAYERS), 2, figsize=(7, 3.3*len(LAYERS)))
    print(f'=== {model_key} ===')
    for row, layer in enumerate(LAYERS):
        acts = cache[f'blocks.{layer}.hook_resid_post'][0]
        if drop_cls: acts = acts[1:]
        try:
            sae = load_sae(model_key, layer, tag)
        except FileNotFoundError as e:
            print('  missing ckpt:', e); continue
        recon = reconstruct(sae, acts)
        idx_o = patch_label_indices(acts,  text, ln=vision.ln_final, projection=proj)
        idx_r = patch_label_indices(recon, text, ln=vision.ln_final, projection=proj)
        agr = float((idx_o == idx_r).float().mean())
        print(f'  layer {layer}: label agreement original vs SAE = {agr:.2%}')
        draw_grid(axes[row,0], img_224, idx_o, label_colors, label_emojis, title=f'{model_key} L{layer} original')
        draw_grid(axes[row,1], img_224, idx_r, label_colors, label_emojis, title=f'{model_key} L{layer} SAE recon (agr {agr:.0%})')
    legend=[mpatches.Patch(color=c,label=f'{e} {n}') for e,n,c in zip(label_emojis,label_names,label_colors)]
    fig.legend(handles=legend, loc='lower center', ncol=len(LABELS), fontsize=8, bbox_to_anchor=(0.5,-0.02))
    plt.suptitle(f'{model_key}: logit lens, original vs SAE-reconstructed', fontsize=11)
    plt.tight_layout(); plt.savefig(f'content/sae_logitlens_{model_key}.png', dpi=140, bbox_inches='tight'); plt.show()


## 5b - Side by side across models (one figure)
Columns: CLIP original, CLIP SAE, SigLIP original, SigLIP SAE. Rows: layers.
Run after the CLIP/SigLIP cell above (reuses its caches, text embeddings, SAEs).

In [ ]:
col_specs = [
    ('CLIP orig',   'clip',   cache_clip, clip_vision,   clip_text,   clip_proj, 'clip_topk_128', True,  False),
    ('CLIP SAE',    'clip',   cache_clip, clip_vision,   clip_text,   clip_proj, 'clip_topk_128', True,  True),
    ('SigLIP orig', 'siglip', cache_sig,  siglip_vision, siglip_text, None,      'topk_128',      False, False),
    ('SigLIP SAE',  'siglip', cache_sig,  siglip_vision, siglip_text, None,      'topk_128',      False, True),
]

fig, axes = plt.subplots(len(LAYERS), 4, figsize=(14, 3.3*len(LAYERS)))
for row, layer in enumerate(LAYERS):
    for col, (name, mkey, cache, vision, text, proj, tag, drop, use_sae) in enumerate(col_specs):
        acts = cache[f'blocks.{layer}.hook_resid_post'][0]
        if drop: acts = acts[1:]
        if use_sae:
            acts = reconstruct(load_sae(mkey, layer, tag), acts)
        idx = patch_label_indices(acts, text, ln=vision.ln_final, projection=proj)
        draw_grid(axes[row, col], img_224, idx, label_colors, label_emojis, title=f'{name} L{layer}')

legend = [mpatches.Patch(color=c, label=f'{e} {n}') for e,n,c in zip(label_emojis, label_names, label_colors)]
fig.legend(handles=legend, loc='lower center', ncol=len(LABELS), fontsize=8, bbox_to_anchor=(0.5, -0.01))
plt.suptitle('SAE reconstruction logit lens, models side by side: CLIP vs SigLIP', fontsize=12)
plt.tight_layout()
plt.savefig('content/sae_logitlens_models_sidebyside.png', dpi=140, bbox_inches='tight')
plt.show()


## 6 — ViT: class-logit readout (no text space)

The supervised ViT has no text encoder, so we read out ImageNet class logits per
patch (final layernorm + classifier head) and report how often the top class is
unchanged after SAE reconstruction. This is the ViT analogue of the agreement number.


In [ ]:
vit_vision = load_hooked_model('vit_base_patch16_224', device=device).to(device); vit_vision.eval()
with torch.no_grad():
    _, cache_vit = vit_vision.run_with_cache(x_vit)

head = getattr(vit_vision, 'head', None) or getattr(vit_vision, 'classifier', None)
lnf  = getattr(vit_vision, 'ln_final', None)

def vit_top_class(acts):
    a = lnf(acts.float()) if lnf is not None else acts.float()
    return head(a).argmax(dim=-1)

print('=== vit (ImageNet class-logit readout) ===')
for layer in LAYERS:
    acts = cache_vit[f'blocks.{layer}.hook_resid_post'][0]
    try:
        sae = load_sae('vit', layer, 'vit_topk_128')
    except FileNotFoundError as e:
        print('  missing ckpt:', e); continue
    recon = reconstruct(sae, acts)
    a_o, a_r = vit_top_class(acts), vit_top_class(recon)
    print(f'  layer {layer}: top-class agreement original vs SAE = {float((a_o==a_r).float().mean()):.2%}')


## 7 — Reading the result

High agreement means the SAE preserved the semantic content the readout sees, so
the sparse features faithfully capture what the model represents at that layer. Low
agreement (expected more at deep layers, where EV was lower) means the sparse
bottleneck dropped information the readout depended on. Compare the agreement trend
across CLIP, SigLIP, and ViT: if one objective's features survive the SAE more
faithfully, that is evidence its representation is more sparsely-structured.
